In [ ]:
import io
import pandas as pd
import scanpy as sc
import pyarrow.dataset as ds
import gcsfs
import numpy as np
import scanpy as sc
import anndata as ad
import h5py
import dask
from collections import Counter
from tqdm import tqdm
import time 

In [ ]:
infile = "/jdata/qmy/VirtualCell/tahoe_100M/plate3_2k-obs.h5ad" 

In [ ]:
with h5py.File(infile, "r") as f:
    adata = ad.AnnData(
        obs=ad.io.read_elem(f["obs"]),
        var=ad.io.read_elem(f["var"]),
    )
    adata.X = ad.io.read_elem(f["X"])
adata = adata[adata.obs["pass_filter"] == "full", :].copy()
# 归一化
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=10000)
highly_variable_genes = set(adata.var_names[adata.var["highly_variable"]])
print(f"识别到 {len(highly_variable_genes)} 个高变基因")
adata = adata[:, list(highly_variable_genes)].copy()
